## Prepare prompts dataset

In [1]:
import requests
from tqdm.auto import tqdm

prompts = None
tries = 10

for i in tqdm(range(tries), desc="Fetching prompts"):
    api_response = requests.get(
        url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj'
    )
    if api_response.ok:
        prompts = api_response.json()
        break

if not prompts:
    raise Exception('Failed to fetch prompts from promptlab API')

print(f"Total prompts fetched: {len(prompts)}")

Fetching prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Total prompts fetched: 365


In [2]:
filtered_prompts = list(
    filter(
        lambda prompt: prompt['status'] == 'APPROVED',
        prompts
    )
)

print(f"Total approved prompts: {len(filtered_prompts)}")

Total approved prompts: 236


In [3]:
filtered_prompts = list(
    filter(
        lambda prompt: prompt['text_direction'] == 'ltr',
        filtered_prompts
    )
)

print(f"Total approved prompts: {len(filtered_prompts)}")

Total approved prompts: 235


In [4]:
filtered_prompts[0]

{'id': 14891,
 'tags': [],
 'name': 'expert Arabic summarizer',
 'task': {'name': 'summarization'},
 'status': 'APPROVED',
 'template': 'You are an expert Arabic text summarizer. The following article:\r\n{{article}}\xa0\r\ncan be summarized as:\r\n|||\r\n{{summary}}',
 'created_by': 'majed.alshaibani',
 'dataset_name': 'arbml/AraSum',
 'dataset_subset': 'default',
 'answer_choices': [],
 'text_direction': 'ltr'}

In [5]:
import datasets
star_templates = datasets.Dataset.from_list(filtered_prompts).remove_columns(['status','created_by'])
star_templates

Dataset({
    features: ['id', 'tags', 'name', 'task', 'template', 'dataset_name', 'dataset_subset', 'answer_choices', 'text_direction'],
    num_rows: 235
})

In [6]:
star_templates = star_templates.map(lambda example: {
    **example,
    'dataset_subset': example['dataset_subset'].strip().split()[0], # Take only the first part of the subset before any space
})

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

In [7]:
from tqdm.auto import tqdm

all_datasets = {
    (prompt['dataset_name'], prompt['dataset_subset']) 
    for prompt in star_templates
}

downloaded_datasets = {}
for dataset_name, dataset_subset in tqdm(all_datasets, desc="Downloading datasets"):
    print('extracting for dataset:', dataset_name, 'with subset:', dataset_subset)
    if dataset_name == 'khalidalt/tydiqa-goldp':
        print('Using streaming load for', dataset_name)
        streamed_dataset = datasets.load_dataset(
            dataset_name,
            dataset_subset,
            trust_remote_code=True,
            streaming=True,
        )
        streamed_dataset = datasets.DatasetDict(
            {
                split: list(streamed_dataset[split])
                for split in streamed_dataset.keys()
            }
        )
        downloaded_datasets[(dataset_name, dataset_subset)] = streamed_dataset
        continue
    downloaded_datasets[(dataset_name, dataset_subset)] = datasets.load_dataset(
        dataset_name,
        dataset_subset,
        trust_remote_code=True,
    )

extracting for dataset: arbml/arabic_text_diacritization with subset: default
extracting for dataset: OALL/Arabic_EXAMS with subset: default
extracting for dataset: arbml/ArEntail with subset: default


Repo card metadata block was not found. Setting CardData to empty.


extracting for dataset: arbml/ArMATH with subset: default
extracting for dataset: arbml/APCDv2 with subset: default
extracting for dataset: arbml/ElecMorocco with subset: default
extracting for dataset: asas-ai/tydiqa-ar with subset: secondary_task
extracting for dataset: arbml/arastance with subset: default
extracting for dataset: arbml/offenseval_2020 with subset: default
extracting for dataset: wiki_lingua with subset: arabic
extracting for dataset: xquad with subset: xquad.ar
extracting for dataset: arbml/ArCovidVac with subset: default
extracting for dataset: arbml/caner_batched with subset: default
extracting for dataset: arbml/OSAC_CNN with subset: default
extracting for dataset: arbml/Commonsense_Validation with subset: default
extracting for dataset: arbml/AT_ODSTA with subset: default
extracting for dataset: khalidalt/tydiqa-goldp with subset: arabic
Using streaming load for khalidalt/tydiqa-goldp
extracting for dataset: arbml/ANS_stance with subset: default
extracting for da

Repo card metadata block was not found. Setting CardData to empty.


extracting for dataset: arbml/AraBench_dev with subset: default
extracting for dataset: arbml/Zero_Shot_Cross_Lingual_NER_ar_batched with subset: default
extracting for dataset: arbml/Ashaar_dataset with subset: default
extracting for dataset: arbml/Arabic_Dialects_Dataset with subset: default
extracting for dataset: arbml/AQMAR_batched with subset: default
extracting for dataset: arbml/Named_Entities_Lexicon with subset: default
extracting for dataset: FahdSeddik/AGS-Corpus with subset: default
extracting for dataset: arbml/Dangerous_Dataset with subset: default
extracting for dataset: arbml/Shami with subset: default


Repo card metadata block was not found. Setting CardData to empty.


extracting for dataset: arbml/BBN_Blog_Posts with subset: default
extracting for dataset: arbml/SANAD with subset: default


Repo card metadata block was not found. Setting CardData to empty.


extracting for dataset: Helsinki-NLP/opus-100 with subset: ar-en
extracting for dataset: arbml/MLMA_hate_speech_ar with subset: default
extracting for dataset: arbml/ACVA with subset: default
extracting for dataset: facebook/belebele with subset: acm_Arab


In [8]:
from collections import defaultdict
splits_dict = defaultdict(list)
for dataset_name, dataset_details in downloaded_datasets.items():
    for split_name, split_data in dataset_details.items():
        splits_dict[f"{split_name}"].append(dataset_name)

The output of the above cell is:
```python
{
    'train': [('arbml/APCDv2', 'default'),
              ('FahdSeddik/AGS-Corpus', 'default'),
              ('arbml/ATT', 'default'),
              ('arbml/antcorpus', 'default'),
              ('arbml/PAAD', 'default'),
              ('arbml/OSAC_CNN', 'default'),
              ('arbml/MLMA_hate_speech_ar', 'default'),
              ('arbml/BBN_Blog_Posts', 'default'),
              ('Helsinki-NLP/opus-100', 'ar-en'),
              ('khalidalt/tydiqa-goldp', 'arabic'),
              ('arbml/Zero_Shot_Cross_Lingual_NER_ar_batched', 'default'),
              ('arbml/ElecMorocco', 'default'),
              ('arbml/Arabic_Hate_Speech', 'default'),
              ('arbml/arabic_text_diacritization', 'default'),
              ('arbml/ANS_stance', 'default'),
              ('ajgt_twitter_ar', 'plain_text'),
              ('arbml/AQMAR_batched', 'default'),
              ('sem_eval_2018_task_1', 'subtask5.arabic'),
              ('arbml/offenseval_2020', 'default'),
              ('arbml/Commonsense_Validation', 'default'),
              ('arbml/Ashaar_dataset', 'default'),
              ('arbml/Arabic_Dialects_Dataset', 'default'),
              ('arbml/Disease_NER_batched', 'default'),
              ('arbml/Twt15DA_Lists', 'default'),
              ('arbml/ArEntail', 'default'),
              ('arbml/Named_Entities_Lexicon', 'default'),
              ('arbml/shakkelha', 'default'),
              ('emotone_ar', 'default'),
              ('arbml/arastance', 'default'),
              ('arbml/Mawqif', 'default'),
              ('arbml/MPOLD', 'default'),
              ('arbml/caner_batched', 'default'),
              ('asas-ai/tydiqa-ar', 'secondary_task'),
              ('arbml/araData', 'default'),
              ('arbml/AT_ODSTA', 'default'),
              ('arbml/oclar', 'default'),
              ('Helsinki-NLP/opus_infopankki', 'ar-en'),
              ('wiki_lingua', 'arabic'),
              ('arbml/NileULex', 'default'),
              ('labr', 'plain_text'),
              ('hard', 'plain_text'),
              ('arbml/ArSarcasm_v2', 'default'),
              ('ar_sarcasm', 'default'),
              ('arbml/ArMATH', 'default'),
              ('arbml/qa4mre', 'default'),
              ('arbml/Dangerous_Dataset', 'default'),
              ('arbml/ArabicTE', 'default'),
              ('arbml/SANAD', 'default'),
              ('GEM/xlsum', 'arabic'),
              ('arbml/AraSum', 'default'),
              ('arbml/ArCovidVac', 'default'),
              ('arbml/APCD_remap', 'default'),
              ('mkqa', 'mkqa'),
              ('arbml/Shami', 'default')],
    'test': [('facebook/belebele', 'acm_Arab'),
              ('Helsinki-NLP/opus-100', 'ar-en'),
              ('OALL/Arabic_EXAMS', 'default'),
              ('arbml/arabic_text_diacritization', 'default'),
              ('arbml/ANS_stance', 'default'),
              ('sem_eval_2018_task_1', 'subtask5.arabic'),
              ('arbml/offenseval_2020', 'default'),
              ('arbml/nsurl_2019_task8_test', 'default'),
              ('arbml/ArEntail', 'default'),
              ('Helsinki-NLP/tatoeba_mt', 'ara-eng'),
              ('arbml/arastance', 'default'),
              ('arbml/ArabicMMLU', 'default'),
              ('arbml/ACVA', 'default'),
              ('labr', 'plain_text'),
              ('arbml/ArSarcasm_v2', 'default'),
              ('arbml/iSarcasmEval_task_A', 'default'),
              ('ar_sarcasm', 'default'),
              ('arbml/CIDAR-MCQ-100', 'default'),
              ('GEM/xlsum', 'arabic')],         
    'validation': [('Helsinki-NLP/opus-100', 'ar-en'),
              ('arbml/AraBench_dev', 'default'),
              ('khalidalt/tydiqa-goldp', 'arabic'),
              ('OALL/Arabic_EXAMS', 'default'),
              ('arbml/Arabic_Hate_Speech', 'default'),
              ('arbml/arabic_text_diacritization', 'default'),
              ('arbml/ANS_stance', 'default'),
              ('sem_eval_2018_task_1', 'subtask5.arabic'),
              ('arbml/Commonsense_Validation', 'default'),
              ('Helsinki-NLP/tatoeba_mt', 'ara-eng'),
              ('arbml/arastance', 'default'),
              ('asas-ai/tydiqa-ar', 'secondary_task'),
              ('arbml/ACVA', 'default'),
              ('xquad', 'xquad.ar'),
              ('GEM/xlsum', 'arabic')]
}
```

## Merge Prompts

In [9]:
from jinja2 import Environment, StrictUndefined
from typing import Dict, List, Optional, Tuple, Any


class TemplateProcessor:

    def __init__(self):
        self.env = Environment(undefined=StrictUndefined)
    
    def apply_template(
        self,
        prompt_template: Dict[str, Any],
        sample: Dict[str, Any],
    ) -> Tuple[bool, str]:

        # Validate template has divider
        template_content = prompt_template['template']
        answer_choices = prompt_template.get('answer_choices', [])
        
        if "|||" not in template_content:
            return False, "Template must contain ||| divider"
        
        # Prepare sample with answer choices
        sample_with_choices = sample.copy()
        if answer_choices:
            sample_with_choices["answer_choices"] = answer_choices
        
        try:
            # Render the template
            template = self.env.from_string(template_content)
            rendered = template.render(**sample_with_choices)
            # Validate answer choices if provided
            if answer_choices:
                validation_error = self._validate_answer_choices(
                    rendered, answer_choices
                )
                if validation_error:
                    return False, validation_error
            
            return True, rendered
            
        except Exception as e:
            return False, f"Error rendering template: {str(e)}"
    
    def _validate_answer_choices(
        self, 
        rendered_template: str, 
        answer_choices: List[str]
    ) -> Optional[str]:
        # Extract the answer part (after |||)
        parts = rendered_template.split("|||")
        if len(parts) < 2:
            return "Template did not produce expected ||| division"
        
        answer_part = parts[-1].strip()
        
        # Check each comma-separated answer
        for answer in answer_part.split(","):
            answer_clean = answer.strip()
            if answer_clean and answer_clean not in answer_choices:
                return f"Output '{answer_clean}' is not in answer_choices {answer_choices}"
        return None

# Convenience function for simple use cases
def apply_template_to_sample(
    prompt: Dict[str, Any],
    sample: Dict[str, Any],
) -> str:
    processor = TemplateProcessor()
    success, result = processor.apply_template(prompt, sample)
    
    if not success:
        raise ValueError(result)
    
    return result

In [ ]:
star = []
failed_prompts = []
for prompt in tqdm(star_templates, desc="Processing prompts"):
    prompt_dataset = (prompt['dataset_name'],prompt['dataset_subset'])
    dataset_splits = downloaded_datasets[prompt_dataset]
    for split_name in dataset_splits.keys():
        for sample in tqdm(dataset_splits[split_name], desc=f"Applying prompt template for prompt: \'{prompt['name']}\' on {split_name} split of {prompt_dataset}", leave=False):
            try:
                merged_prompt = dict(prompt)
                merged_prompt['split_name'] = split_name
                merged_prompt['full_instruction'] = apply_template_to_sample(merged_prompt, sample)
                prompt_template = merged_prompt.pop('template')
                merged_prompt['instruction_template'] = prompt_template
                merged_prompt['instruction_name'] = merged_prompt.pop('name')
                merged_prompt['instruction_tasks'] = list(merged_prompt.pop('task').values())
                splitted_instruction = merged_prompt['full_instruction'].split('|||')
                # assert the occurances of '|||' in the template is one, important for splitting into instructions input and instructions output
                if len(splitted_instruction) != 2:
                    failed_prompts.append(merged_prompt)
                    print(f'found that this prompt (id:{merged_prompt["id"]}) contains ', len(splitted_instruction), ' parts after splitting by |||, adding to failed_prompts list..')
                    continue
                merged_prompt['instruction_input'] = splitted_instruction[0].strip()
                merged_prompt['instruction_output'] = splitted_instruction[1].strip()
                merged_prompt['instruction_template_id'] = merged_prompt.pop('id')
                star.append(merged_prompt)
            except Exception as e:
                print(f"Error processing sample with prompt '{prompt['name']}' and dataset {prompt_dataset} on split {split_name}'")
                print('prompt id is ', prompt['id'])
                print(f"Error details: {e}")
                raise e
len(star)

Processing prompts:   0%|          | 0/235 [00:00<?, ?it/s]

Applying prompt template for prompt: 'expert Arabic summarizer' on train split of ('arbml/AraSum', 'default'):…

Applying prompt template for prompt: 'Translation as completion' on test split of ('Helsinki-NLP/tatoeba_mt', …

Applying prompt template for prompt: 'Translation as completion' on validation split of ('Helsinki-NLP/tatoeba…

Applying prompt template for prompt: 'Simple and direct instruction' on test split of ('Helsinki-NLP/tatoeba_m…

...

Applying prompt template for prompt: 'MaisPrompt' on test split of ('arbml/ACVA', 'default'):   0%|          |…

Applying prompt template for prompt: 'MathsPrompt' on train split of ('arbml/ArMATH', 'default'):   0%|       …

Applying prompt template for prompt: 'Simple Meter Classification English Prompt 2' on train split of ('arbml/…

Applying prompt template for prompt: 'Simple Meter Classification Arabic Prompt' on train split of ('arbml/APC…

Applying prompt template for prompt: 'Meter Classification English Prompt with Explanation' on train split of …

In [ ]:
star[0]

{'tags': [],
 'dataset_name': 'arbml/AraSum',
 'dataset_subset': 'default',
 'answer_choices': [],
 'text_direction': 'ltr',
 'split_name': 'train',
 'full_instruction': 'You are an expert Arabic text summarizer. The following article:\n"بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ميونيخ محاكمة رجل من جمهورية الجبل الأسود ( مونتينيغرو)، اعتقل في ألمانيا للاشتباه في انه كان ينقل أسلحة قبل أيام من اعتداءات باريس التي حصلت في تشرين الثاني/نوفمبر العام الماضي. وتريد المحكمة معرفة عما إذا كانت هذه الأسلحة أعدت للاستخدام في هجمات فرنسا. وقالت المتحدثة إن المتهم ""اعترف أنه كان على علم بوجود أسلحة في سيارته، لكنه لا يعرف ما إذا كانت ستستخدم لتنفيذ اعتداء"". وأضافت المتحدثة أن الرجل الذي يبلغ الحادية والخمسين ويدعى فوسيليتش، ملاحق ""بتهمة التآمر في الإعداد لأعمال عنف يهدد امن الدولة"" الفرنسية. واعتقل الرجل في 5 تشرين الثاني/نوفمبر 2015 على طريق سريع في ولاية بافاريا، على مقربة من الحدود النمساوية. حيث عثر رجال الشرطة في سيارته على كمية من الأسلحة و عدد كبير من القنابل اليدوية وبنادق كالاشني